In [1]:
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from langchain_community.retrievers import BM25Retriever
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import re
import pickle
from langchain_openai import OpenAIEmbeddings
import faiss
from pathlib import Path
import sys

In [2]:
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
META_FILE   = PROJECT_ROOT / "data" / "raw" / "meta_Toys_and_Games.jsonl"
REVIEW_FILE = PROJECT_ROOT / "data" / "raw" / "Toys_and_Games.jsonl"

meta = pd.read_json(META_FILE, lines=True, nrows=100000)
review = pd.read_json(REVIEW_FILE, lines=True, nrows=100000)

<div class="alert alert-info">
    
### Step 1: Data Exploration and Preprocessing
</div>

In [3]:
review.dtypes

rating                        int64
title                           str
text                            str
images                       object
asin                            str
parent_asin                     str
user_id                         str
timestamp            datetime64[ms]
helpful_vote                  int64
verified_purchase              bool
dtype: object

In [4]:
meta.dtypes

main_category          str
title                  str
average_rating     float64
rating_number        int64
features            object
description         object
price              float64
images              object
videos              object
store                  str
categories          object
details             object
parent_asin            str
bought_together    float64
subtitle               str
author              object
dtype: object

In [5]:
cleaned_meta = meta.drop(columns = ['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'])
cleaned_meta.head()

,main_category,title,average_rating,rating_number,features,description,store,categories,details,parent_asin
0,Toys & Games,"KUNGOON Happy Anniversary Balloon Banner,Weddi...",4.5,241,[],[],Kunggo,[],{'Package Dimensions': '10.12 x 8.03 x 0.51 in...,B08GPM7CQN
1,Toys & Games,Gothic Mothman Plushie Doll with Bright Red Ey...,1.3,2,[🦋 Mothman’s bright red eyes could stare you d...,[🦋 Description: Mothman’s bright red eyes coul...,Felicy,"[Toys & Games, Stuffed Animals & Plush Toys, P...","{'Item Weight': '2.47 ounces', 'Manufacturer r...",B09X9XW42H
2,Toys & Games,Melody Jane Dollhouse Builders DIY 1:24 Scale ...,4.2,67,[1:24 Scale - Plastic - Approximate cut out si...,[],Melody Jane Dolls Houses,"[Toys & Games, Dolls & Accessories, Dollhouse ...","{'Item Weight': '0.48 ounces', 'Manufacturer r...",B01I9QET6M
3,Toys & Games,Traxxas Stampede 4X4: 1/10 Scale 4wd Monster T...,4.5,48,[Waterproof electronics for all-weather drivin...,[Stampede 4X4 is built Traxxas Tough to withst...,Traxxas,"[Toys & Games, Remote & App Controlled Vehicle...",{'Product Dimensions': '15.63 x 13.39 x 8.94 i...,B019XEEX1A
4,Toys & Games,Hot Wheels Monster Truck 1:24 Scale 2022 Bone ...,4.8,17699,[Designed in 1:24 scale with durable die-cast ...,[The Hot Wheels Monster Trucks 1:24 scale die-...,Hot Wheels,"[Toys & Games, Preschool, Pre-Kindergarten Toys]",{'Product Dimensions': '5 x 6.27 x 5.5 inches'...,B09G7K3JWQ


In [6]:
reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns = ['images', 'timestamp', 'user_id', 'verified_purchase'])
cleaned_reviews.head()

,rating,title,text,asin,parent_asin,helpful_vote
0,5,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,B09QH7QJS7,0
1,3,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,B06XYKSKQP,1
2,5,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07SFF3YQW,B07XRSD5R9,0
3,5,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,B007JWWUDW,0
4,3,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,B00MZG6OO8,2


In [7]:
parent_asin = list(pd.Series(np.intersect1d(cleaned_reviews['parent_asin'], cleaned_meta['parent_asin'])))

cleaned_reviews = cleaned_reviews[cleaned_reviews['parent_asin'].isin(parent_asin)]
cleaned_meta = cleaned_meta[cleaned_meta['parent_asin'].isin(parent_asin)]

In [8]:
cleaned_meta['description'] = cleaned_meta['description'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
)
cleaned_meta['description']

0                                                         
4        The Hot Wheels Monster Trucks 1:24 scale die-c...
9                     Tamiya 1/35 Brick Wall Set, TAM35028
21                    LEGO Minifigures Series 11 Scarecrow
30       Guardhouse heavy duty coin boxs are crafted of...
                               ...                        
99941    Our new full body, performance quality, ventri...
99945                                                     
99952    From teaching first words to encouraging first...
99973    This soft, cuddly bunny figure wants to belong...
99985    Barbie doll and her cake decorating playset ar...
Name: description, Length: 7435, dtype: str

In [9]:
cleaned_meta['details'] = cleaned_meta['details'].apply(
    lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
)
cleaned_meta['details']

0        Package Dimensions 10.12 x 8.03 x 0.51 inches ...
4        Product Dimensions 5 x 6.27 x 5.5 inches Item ...
9        Brand TAMIYA Age Range (Description) Kid Theme...
21       Product Dimensions 0.5 x 0.5 x 1 inches Item W...
30       Brand Guardhouse Material Engineered Wood Prod...
                               ...                        
99941    Product Dimensions 25 x 12 x 10 inches Item mo...
99945    Material Polyester Brand Jay Franco Style Mode...
99952    Product Dimensions 21.26 x 16.69 x 18.11 inche...
99973    Product Dimensions 7.87 x 8.66 x 8.27 inches I...
99985    Product Dimensions 3 x 15.5 x 12.75 inches Ite...
Name: details, Length: 7435, dtype: str

In [10]:
cleaned_meta['features'] = cleaned_meta['features'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)
cleaned_meta['features']

0                                                         
4        Designed in 1:24 scale with durable die-cast m...
9        This plastic model kit requires plastic cement...
21                                Packaged in zip lock bag
30       This chipboard box stores up to 24 proof sets ...
                               ...                        
99941    High quality full body ventriloquist style pup...
99945                                            Polyester
99952    Interactive baby walker and learning toy with ...
99973    Having your own bunny figure to love is the be...
99985    Barbie baker doll comes with a sweet playset t...
Name: features, Length: 7435, dtype: str

In [11]:
cleaned_meta['categories'] = cleaned_meta['categories'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)
cleaned_meta['categories']

0                                                         
4             Toys & Games Preschool Pre-Kindergarten Toys
9                                                         
21                      Toys & Games Building Toys Figures
30                                                        
                               ...                        
99941               Toys & Games Puppets & Puppet Theaters
99945                                                     
99952    Toys & Games Preschool Toddler Toys Push & Pul...
99973       Toys & Games Kids' Electronics Electronic Pets
99985            Toys & Games Dolls & Accessories Playsets
Name: categories, Length: 7435, dtype: str

##### Description of text preprocessing decisions
To prepare the data, some preprocessing steps were taken in order to ensure there is consistency and usability of textual fields.
First, only relevant and needed columns were kept from both datasets. From the metadata, we only kept columns such as `title`, `features`, `description`, and `categories`, while other columns that were less informative and noisy were removed. These removed columns ranged from `images` to `videos` and `prices`.

As the next step, the two datasets were aligned by filtering on the intersection of the `parent_asin` to make sure that only the products that appear in both datasets are included.

Furthermore, due to the inconsistency of the text fields, normalization was required. Specifically, the `description` column sometimes appeared as a list or dictionary, so flattening it into a single string was the appropriate step.
The `features` column, originally a list, was converted into a string separated by space. Lastly, the `details` column, which was a dictionary, was turned into a string by joining them via key–value pairs.

Through these EDA steps, the text information in the dataset became consistent, flat strings that were more suitable for the tokenization and other things that are going to take place in BM25 and embedding-based search.

<div class="alert alert-info">
    
### Step 2 and 3: BM25 Search and Semantic Search
</div>

In [ ]:
# code adapted from lecture 5

# combining all the important and useful columns from the meta dataset
products = (
    cleaned_meta['title'].fillna('') + ' ' +
    cleaned_meta['description'].fillna('') + ' ' +
    cleaned_meta['features'].fillna('') + ' ' +
    cleaned_meta['categories'].fillna('')
)
products = products.tolist()
queries = cleaned_reviews['title'].tolist()

def simple_tokenize(text):
    """Tokenization by lowercasing, stripping non-alphanumeric characters, 
    and splitting text into tokens."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", "", text)
    return text.split()

tokenized_products = [simple_tokenize(p) for p in products]
bm25 = BM25Okapi(tokenized_products)

model = SentenceTransformer("all-MiniLM-L6-v2")
product_embeddings = model.encode(products).astype("float32")

# using faiss to index product embeddings for semantic search
index = faiss.IndexFlatL2(product_embeddings.shape[1])
index.add(product_embeddings)

def bm25_search(query, top_k=3):
    """Return the top-k products ranked by BM25 relevance score for a given query."""
    tokenized_query = simple_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [(products[i], scores[i]) for i in ranked_idx]

def embedding_search(query, top_k=3):
    """Retrieve the top-k products by semantic similarity using FAISS vector search."""
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [(products[i], distances[0][rank]) for rank, i in enumerate(indices[0])]

for q in queries[:2]:
    print("=" * 80)
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    for rank, (product, score) in enumerate(bm25_search(q), start=1):
        print(f"{rank}. ({score:.3f}) {product}")

    print("\nEmbedding search top results:")
    for rank, (product, score) in enumerate(embedding_search(q), start=1):
        print(f"{rank}. ({score:.3f}) {product}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: super cute. my 4 year old loves

BM25 top results:
1. (14.056) TOP BRIGHT Toddler Fishing Game for 2 Year Old, Kids Fishing Games for 2 Year Olds, Toddler Birthday Gift for Two Year Old Boy Girl  COGNITIVE DEVELOPMENT TOYS: Let your child enjoy hours of fun with the TOP BRIGHT toddler fishing game. The fishing game for 2 year old includes 26 wooden fish puzzles which are patterned with alphabet, numbers and cute ocean creatures, making it a great teaching tool to develop their hand and eye coordination as well as their motor skills. It’s never too early to begin STEM educational techniques HOW TO PLAY: The aim of the fishing game is to catch 26 wooden fish with the aide of magnetic rods. Your kids can play with an adult or a friend as they follow their natural instinct and explore the fun of a pretend play game whilst learning to play fairly and share. Playing with someone else also develops their social skills which makes it a must have two year old boy gifts BRIGHT, COLORFUL A

In [13]:
bm25_PATH = PROJECT_ROOT / "data" / "processed" / "bm25_index.pkl"
with open(bm25_PATH, "wb") as i:
    pickle.dump(bm25, i)

tokenized_products_PATH = PROJECT_ROOT / "data" / "processed" / "tokenized_products.pkl"
with open(tokenized_products_PATH, "wb") as i:
    pickle.dump(tokenized_products, i)

In [14]:
product_index_PATH = PROJECT_ROOT / "data" / "processed" / "product_index.faiss"
faiss.write_index(index, str(product_index_PATH))

<div class="alert alert-info">
    
### Step 4: Qualitative Evaluation
</div>

In [15]:
query_set = {
    "keyword-based": [
        "Christmas toys for kids",
        "Action games for boys",
        "16 oz bottle"
    ],
    "semantic-based": [
        "Toys for kids at least 5 years old",
        "Games that boost children's creativity",
        "Toy or game for kids who love animals"
    ],
    "complex": [
        "Games that are straightforward to learn",
        "Sturdy and durable toys that can last long",
        "Games that are not easy for kids to become addicted",
        "Toys that can be played with a dog"
    ]
}

In [16]:
for key, val in query_set.items():
    for query in val:
        print("=" * 80)
        print(f"QUERY: {query} ({key}) \n")

        print("BM25 top 5 results:")
        for rank, (product, score) in enumerate(bm25_search(q, top_k=5), 
                                                start=1):
            print(f"{rank}. ({score:.3f}) {product}")

        print("\nEmbedding search top 5 results:")
        for rank, (product, score) in enumerate(embedding_search(q, top_k=5), 
                                                start=1):
            print(f"{rank}. ({score:.3f}) {product}")
        print()

QUERY: Christmas toys for kids (keyword-based) 

BM25 top 5 results:
1. (15.373) Fire Fox EPO Foam Super Durable 21.75" Hand Glider , White Firefox hand glider is perfect for the back yard, front yard and parks. It will glide for over 75 feet. Made of EPO foam, which is very flexible, makes it almost indestructible. Careful use should yield hours and hours of flying fun. No Glue or Tape required. Wingspan 22.75". Made from EPO Foam Glides 75 Feet Wingspan 21.75" Firefox hand glider is perfect for the back yard, front yard and parks It will glide for over 75 feet Made of EPO foam, which is very flexible, makes it almost indestructible Careful use should yield hours and hours of flying fun No Glue or Tape Required Toys & Games Novelty & Gag Toys Flying Toys Airplane Construction Kits
2. (11.610) smttw Wooden Sorting & Stacking Toys for 1-3 Toddlers, Montessori Toys Educational Shape Sorter Color Recognition Toys for Boys Girls, Preschool Kids Learning Puzzle Gift(4 Shapes 20 Pieces)  Ent

<div class="alert alert-info">
    
### Step 5: Web App
</div>

Please find the locally running app from this file: `app/app.py`